# 01 · RAAMove — build the pool

*Rhetorical moves in research-article abstracts (8 classes)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** 400 RA abstracts, annotated sentence by sentence with one of eight rhetorical moves (Background, Gap, Purpose, Method, Result, Conclusion, Contribution, Implication). Reported annotator agreement: κ = 0.785.

**Difficulty of the labeling judgment:** ★★☆ — moderate. Moves are functional categories, so neighbouring sentences can be genuinely hard to separate.

**Licence:** CC BY 4.0  
**Cite:** Liu, J. et al. (2024), *LREC-COLING*. github.com/ljk1228/RAAMove

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

Those three keys are required on every track. Two tracks add more: `cars50` and `raamove` ask what a sentence *does in a passage*, which is not always decidable from the sentence on its own, so their items also carry `doc_id`, `sent_index`, `n_sents` and `context`. Extra keys are safe everywhere — nothing in the pipeline checks for keys it does not need.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work in the RUNTIME, not in Drive: the next cells download a whole
    # corpus, and raw data is big, mostly not ours to redistribute, and one
    # command to fetch again. The pool you build from it is what persists.
    os.makedirs("/content/raw", exist_ok=True)
    os.chdir("/content/raw")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, MEMBERS,
                    LABELS_ORDER, ROOT, OUT_DIR, POOL_PATH, DEMO_POOL_PATH,
                    SAMPLE_PATH, GOLD_PATH, PRED_PATH, ROUNDS_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)

describe()                  # what this notebook is working on


## Step 1 — Download the raw data

In [ ]:
!git clone --depth 1 https://github.com/ljk1228/RAAMove

## Step 2 — Look at the raw format

This one is **JSON**, split into two files by discipline (`Intelligence.json`, `Engineering.json`). Each record has a `text`, a three-letter move code in `labels`, and an `idx` — the number of the abstract the sentence came from.

**Read the codes the cell below prints** — you are about to name every one. Then look at the `idx` values: the file is one long list of sentences, but the sentences of an abstract sit together, in order. That is the only reason the abstracts can be put back together at all.

In [ ]:
RAW_DIR = "RAAMove"

import json
from collections import Counter

data = json.loads(open(RAW_DIR + "/Intelligence.json", encoding="utf-8").read())
print("records:", len(data))
print("codes:", Counter(record["labels"] for record in data))
print("abstracts:", len({record["idx"] for record in data}))
data[:3]

### Reading JSON with `json.loads`

`json.loads(text)` turns a JSON string into ordinary Python objects — a JSON array becomes a `list`, a JSON object becomes a `dict`. Nothing else is needed: once it is loaded you index it exactly like any other list of dicts, which is what the cell above did.

**One record looks like this:**

```json
{"idx": 0, "text": "Recent work has shown ...", "labels": "BAC"}
```

* `record["text"]` — the sentence from the abstract
* `record["labels"]` — the move, as a three-letter code (singular value, despite the plural key)
* `record["idx"]` — which abstract it came from. **Careful:** `idx` starts again at 0 in the second file, so it is not on its own a unique id.

The reshaping function below uses exactly these:

1. `path.read_text()` then `json.loads(...)` — the file as a list of records.
2. `record["labels"]` — the raw code, e.g. `BAC`.
3. `RAAMOVE_LABELS[code]` — your dict, turning that code into the move name your prompt will use.
4. `record["idx"]` — used to group the flat list back into abstracts, so each sentence can carry the one it came from. See step 3.
5. The two discipline files are read into one pool — an assumption, not a fact. See the note in step 3.

## Step 3 — Reshape into the canonical schema

Two decisions:

1. ✏️ **What each code is called.** `BAC` → `Background`. The expansion is not cosmetic: it is the wording your prompt will use, your annotators will read on the sheet, and your confusion matrix will be labelled with. `Gap` and `Establishing a niche` describe the same category and will not get you the same predictions.
2. **Pool the two disciplines** — the code below reads both files into one set, treating a move as a rhetorical function rather than a discipline-specific one. That *is* an assumption. It is in the code rather than in a ✏️ cell only because unpicking it makes a better extension than a starting point: comparing Intelligence against Engineering separately would be a real finding.

### Each sentence keeps its abstract

A move is not a property of a sentence. *"We used a mixed-effects model"* is a `Method`; *"the mixed-effects model showed no effect"* is a `Result`; and plenty of real sentences sit between the two and are settled only by what came before them. So the code below does not throw the abstract away. It reads each file **twice** — once to group the flat list of sentences back into abstracts using `idx`, once to emit the items — and every item comes out carrying four extra fields on top of the canonical three:

| field | what it is |
|---|---|
| `doc_id` | which abstract, e.g. `Intelligence-0` |
| `sent_index` | where in it, counting from 0 |
| `n_sents` | how many sentences the abstract has |
| `context` | the abstract itself, one sentence per line |

Those fields travel with the item all the way: notebook 03 shows the abstract to your two coders, and notebook 04 can put it in the prompt. Whether you *use* it is a decision for `PLAN.md` — `prompts/raamove.txt` shows the model the sentence alone, `prompts/raamove_context.txt` shows it the abstract first, and running both is one of the cleanest experiments this track offers.

In [ ]:
# ✏️ Step 3a · Name the moves ────────────────────────────────────
# Goal      : fill in the move name your prompt will use for each three-letter code.
# Shape     : the eight codes are given — you write the eight names
#             RAAMOVE_LABELS = {"BAC": "Background", "GAP": "Gap", ...}
# Produce   : RAAMOVE_LABELS (a dict)      ← later cells use this name
# Note      : the names are not cosmetic. They are the wording your prompt
#             uses, your coders read on the sheet, and your confusion matrix
#             is labelled with. "Gap" and "Establishing a niche" name the
#             same category and will not get you the same predictions.
# Note      : eight classes is a lot. To MERGE two, give them the same name
#             — {"RST": "Finding", "CLN": "Finding"} makes one class of
#             two. Decide that HERE and say so in PLAN.md, not after you
#             have seen the model do badly on them.

# ✏️ your code here — fill in each ____

RAAMOVE_LABELS = {
    "BAC": ____,      # e.g. "Background" — in quotes: the wording your
                      #   prompt, your sheet and your matrix will use
    "GAP": ____,
    "MTD": ____,
    "PUR": ____,
    "RST": ____,
    "CLN": ____,
    "CTN": ____,
    "IMP": ____,
}

print(RAAMOVE_LABELS)


The function below reads the `RAAMOVE_LABELS` you just defined.

In [ ]:
import json
from pathlib import Path

def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1

    for item in items:
        ### Copy before writing ###
        new_item = dict(item)                    # Work on a copy, so the caller's item is left alone.

        ### Stamp the id ###
        new_item["id"] = next_id                 # Overwrite whatever id was there with the running number.
        renumbered.append(new_item)              # Keep it in the order it arrived.
        next_id = next_id + 1                    # Advance, so the next item gets a fresh id.

    return renumbered

def reshape_raamove(raamove_dir):
    """Read RAAMove's per-domain JSON files and expand the 3-letter move codes.

    The corpus ships two domains (Intelligence, Engineering) as separate files. We pool
    them, because a move is meant to be a rhetorical function rather than a
    discipline-specific one - but that IS an assumption, and comparing the two domains
    separately would be a perfectly good extension.

    A move is a rhetorical function WITHIN an abstract, so each item also carries the
    abstract it came from - see the note on the two-pass loop below.
    """
    source_dir = Path(raamove_dir)
    rows = []

    ### Read both discipline files into one pool ###
    for filename in ("Intelligence.json", "Engineering.json"):
        path = source_dir / filename
        if not path.exists():                    # A missing file is not fatal; use whichever shipped.
            continue

        ### Parse the JSON ###
        data = json.loads(path.read_text(encoding="utf-8"))   # -> a list of {"idx": ..., "text": ..., "labels": ...} records.

        ### PASS 1: group the flat record list back into abstracts ###
        # The file is one long list of sentences, but `idx` is the abstract number, and
        # the sentences of one abstract sit together in reading order. So a new idx means
        # a new abstract - which is all the grouping we need, and it does not care that
        # idx starts again at 0 in the other discipline file.
        abstracts = []
        for record in data:
            if not abstracts or abstracts[-1][0] != record["idx"]:
                abstracts.append((record["idx"], []))          # Start collecting a new abstract.
            abstracts[-1][1].append(record)                    # Same idx: same abstract as the line before.

        ### PASS 2: emit one item per sentence, with its abstract attached ###
        for number, records in abstracts:
            texts = [record["text"].strip() for record in records]   # The abstract, sentence by sentence.
            context = "\n".join(texts)           # One string, newlines kept so the sentences stay visible.
            doc_id = path.stem + "-" + str(number)   # e.g. "Intelligence-0". idx alone is NOT unique across the two files.

            for position, record in enumerate(records):
                code = record["labels"]          # e.g. "BAC".
                if code in RAAMOVE_LABELS:
                    label = RAAMOVE_LABELS[code]     # The name your prompt and annotation sheet will use.
                else:
                    label = code        # an unexpected code: keep it and let validate() complain
                rows.append({"id": 0, "text": texts[position], "label": label,
                             "doc_id": doc_id,           # Which abstract this sentence is from.
                             "sent_index": position,     # Where in it - 0 is the first sentence.
                             "n_sents": len(texts),      # How long the abstract is.
                             "context": context})        # The abstract itself.

    return reid(rows)                            # Hand back with ids running 1..N.

def validate(items, allowed=None):
    """Check the canonical schema, and raise on the first problem found.

    Deliberately explicit rather than `assert`: assertions vanish under `python -O`,
    and a silently unvalidated dataset is exactly the kind of thing that surfaces as a
    baffling metric three days later.
    """
    seen_ids = set()
    for position, item in enumerate(items):
        missing = {"id", "text", "label"} - set(item)
        if missing:
            raise ValueError("item " + str(position) + " is missing "
                             + str(sorted(missing)) + ": " + repr(item))
        if item["id"] in seen_ids:
            raise ValueError("duplicate id " + str(item["id"]))
        seen_ids.add(item["id"])
        if not isinstance(item["text"], str) or not item["text"].strip():
            raise ValueError("empty text at id " + str(item["id"]))
        if not isinstance(item["label"], str) or not item["label"]:
            raise ValueError("empty label at id " + str(item["id"]))
        if allowed is not None and item["label"] not in allowed:
            raise ValueError("label " + repr(item["label"]) + " at id "
                             + str(item["id"]) + " is not in " + str(sorted(allowed)))

In [ ]:
rows = reshape_raamove(RAW_DIR)

## Step 4 — Check the label balance

Very imbalanced: `Method` is the biggest class by far, and `Implication` has only a couple of dozen sentences.

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
print("fields per item:", list(rows[0]))

# Peek at the first three. `context`, where a track has one, is trimmed: it is
# the whole passage and would bury everything else in this output.
for item in rows[:3]:
    preview = dict(item)
    if preview.get("context"):
        preview["context"] = preview["context"][:70] + " …"
    print(preview)

## Step 5 — Save it

In [ ]:
# Save the pool into your group's Drive folder, under the exact name notebook
# 02 will look for. POOL_PATH comes from config.yaml, so the two cannot drift.
import json

if TRACK not in ['raamove']:
    raise RuntimeError(
        "config.yaml says  track: " + str(TRACK) + "  but this is the raamove "
        "notebook, so saving now would put raamove data into "
        + POOL_PATH.name + ", which belongs to another track.\n"
        "Open config.yaml, set  track: to one of raamove,"
        " save it, then re-run the SETUP cell at the top of this notebook.")


# Check the shape before writing. Everything downstream — the sampling, the
# annotation sheet, the scoring — assumes every item has an id, a text and a
# label, and a pool that breaks that assumption does not fail here: it fails
# in notebook 03, after two people have annotated forty items.
validate(rows)

POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(POOL_PATH, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", POOL_PATH)

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** open `02_sample.ipynb`. It reads `POOL_PATH` — the file the cell above just wrote, in your group's Drive folder. Nothing to copy, nothing to paste: that path is the handoff, and both notebooks get it from the same `config.yaml`.